# Banking Incident Classification: v1

**Question 1:** Can critical incident hours be identified from operational KPIs in the same hour?
**Question 2:** Does the previous hour's KPIs give early warning of a critical hour?

**Data (real files only):**
- `incident_reports.csv`: 168 hourly rows, 2025-10-20 to 2025-10-26. Label: `is_critical` (26 critical, 142 normal).
- `infrastructure_metrics.csv`: 10,080 minute-level rows for the same week. Used for data checks and a file comparison.

**Scope:** One labeled dataset, one target, time-based evaluation. Log clustering, forecasting, RAG, FastAPI, and Docker are future work and are not part of this notebook.

## 1. Load and check the data

In [1]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

inc = pd.read_csv('incident_reports.csv', parse_dates=['timestamp']).sort_values('timestamp').reset_index(drop=True)
infra = pd.read_csv('infrastructure_metrics.csv', parse_dates=['timestamp']).sort_values('timestamp').reset_index(drop=True)

KPIS = ['cpu_utilization', 'memory_usage', 'db_connection_pool_usage', 'application_error_rate', 'neft_success_rate']
NO_NEFT = [k for k in KPIS if k != 'neft_success_rate']
TARGET = 'is_critical'

print('Incident file:', inc.shape, '| time range:', inc.timestamp.min(), 'to', inc.timestamp.max())
print('Infrastructure file:', infra.shape, '| time range:', infra.timestamp.min(), 'to', infra.timestamp.max())
print('Missing values (incident):', int(inc.isna().sum().sum()), '| (infrastructure):', int(infra.isna().sum().sum()))
print('Duplicate timestamps (infrastructure):', int(infra.timestamp.duplicated().sum()))
print('Gaps in infrastructure (not 1 min):', int((infra.timestamp.diff().dropna() != pd.Timedelta('1min')).sum()))
print('Gaps in incident (not 1 h):', int((inc.timestamp.diff().dropna() != pd.Timedelta('1h')).sum()))
print('Label counts:', inc[TARGET].value_counts().to_dict())

FileNotFoundError: [Errno 2] No such file or directory: 'incident_reports.csv'

## 2. Normal vs. critical hours

In [ ]:
print('Mean KPI by label:')
print(inc.groupby(TARGET)[KPIS].mean().round(2).to_string())
print()
rows = []
for f in KPIS:
    n, c = inc.loc[inc[TARGET]==0, f], inc.loc[inc[TARGET]==1, f]
    if c.min() > n.max(): rows.append((f, f'critical if > {n.max():.2f}', 'yes'))
    elif c.max() < n.min(): rows.append((f, f'critical if < {n.min():.2f}', 'yes'))
    else: rows.append((f, 'overlaps', 'no'))
print(pd.DataFrame(rows, columns=['KPI', 'single-KPI rule', 'separates perfectly']).to_string(index=False))

Mean KPI by label:
             cpu_utilization  memory_usage  db_connection_pool_usage  application_error_rate  neft_success_rate
is_critical                                                                                                    
0                      48.87         43.90                     39.28                    2.05              98.04
1                      89.84         85.37                     89.59                   14.54              87.32

                     KPI     single-KPI rule separates perfectly
         cpu_utilization critical if > 76.38                 yes
            memory_usage critical if > 66.26                 yes
db_connection_pool_usage critical if > 66.58                 yes
  application_error_rate  critical if > 6.00                 yes
       neft_success_rate critical if < 96.00                 yes


**Reading:** Every KPI separates critical from normal hours by itself. This means the labels are very likely rule-based, so a model that uses these KPIs can score perfectly without learning anything beyond that rule.

## 3. The two data files disagree

In [ ]:
joined = inc.set_index('timestamp').join(infra.set_index('timestamp')[['neft_success_rate']], rsuffix='_infra', how='inner')
match = (joined.neft_success_rate - joined.neft_success_rate_infra).abs() < 0.01
crit = joined[TARGET] == 1
print(f'Hours compared: {len(joined)} | matching NEFT values: {int(match.sum())}')
print(f'Critical hours: {int(crit.sum())} | critical hours that match: {int((match & crit).sum())}')
print()
print('At critical hours (mean):')
print(joined.loc[crit, ['neft_success_rate', 'neft_success_rate_infra']].mean().round(2).to_string())

Hours compared: 168 | matching NEFT values: 142
Critical hours: 26 | critical hours that match: 0

At critical hours (mean):
neft_success_rate          87.32
neft_success_rate_infra    98.12


**Reading:** The files agree at 142 hours and disagree at all 26 critical hours. The incident file shows NEFT near 87%, the minute-level file near 98%. v1 uses the incident file. Which one is correct is an open question for the instructor.

## 4. Time-based split

In [ ]:
cutoff = inc.timestamp.min() + pd.Timedelta(days=5)
train, test = inc[inc.timestamp < cutoff], inc[inc.timestamp >= cutoff]
print(f'Train: {len(train)} hours, {int(train[TARGET].sum())} critical')
print(f'Test:  {len(test)} hours, {int(test[TARGET].sum())} critical')

Train: 120 hours, 17 critical
Test:  48 hours, 9 critical


## 5. Same-hour detection

In [ ]:
def score(name, y_true, y_pred):
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)
    print(f'{name}: precision={p:.3f} recall={r:.3f} F1={f:.3f}')
    print(confusion_matrix(y_true, y_pred), '\n')
    return {'test': name, 'precision': round(p,3), 'recall': round(r,3), 'f1': round(f,3)}

def logreg():
    return make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000))

results = []
y = test[TARGET]
results.append(score('Same hour: NEFT < 95% rule', y, (test.neft_success_rate < 95).astype(int)))
for name, feats in [('Same hour: logistic, all KPIs', KPIS), ('Same hour: logistic, without NEFT', NO_NEFT)]:
    m = logreg().fit(train[feats], train[TARGET])
    results.append(score(name, y, m.predict(test[feats])))

Same hour: NEFT < 95% rule: precision=1.000 recall=1.000 F1=1.000
[[39  0]
 [ 0  9]] 

Same hour: logistic, all KPIs: precision=1.000 recall=1.000 F1=1.000
[[39  0]
 [ 0  9]] 

Same hour: logistic, without NEFT: precision=1.000 recall=1.000 F1=1.000
[[39  0]
 [ 0  9]] 



**Reading:** Every same-hour method gets all 48 test hours right. This matches the single-KPI finding in section 2. It shows the labels can be recovered from the KPIs, not that a model has learned an early-warning pattern.

## 6. Early warning: previous hour predicts this hour

In [ ]:
lagged = inc[KPIS].shift(1).add_suffix('_prev')
d = pd.concat([inc[['timestamp', TARGET]], lagged], axis=1).dropna().reset_index(drop=True)
tr, te = d[d.timestamp < cutoff], d[d.timestamp >= cutoff]
lag_feats = [c for c in d.columns if c.endswith('_prev')]
print(f'Train: {len(tr)} rows, {int(tr[TARGET].sum())} critical | Test: {len(te)} rows, {int(te[TARGET].sum())} critical')
print()

results.append(score('Early warning: previous hour NEFT < 95%', te[TARGET], (te.neft_success_rate_prev < 95).astype(int)))
m = logreg().fit(tr[lag_feats], tr[TARGET])
results.append(score('Early warning: logistic, lagged KPIs', te[TARGET], m.predict(te[lag_feats])))

pd.DataFrame(results).set_index('test')

Train: 119 rows, 17 critical | Test: 48 rows, 9 critical

Early warning: previous hour NEFT < 95%: precision=0.000 recall=0.000 F1=0.000
[[30  9]
 [ 9  0]] 



Early warning: logistic, lagged KPIs: precision=0.214 recall=0.333 F1=0.261
[[28 11]
 [ 6  3]] 



,precision,recall,f1
test,,,
Same hour: NEFT < 95% rule,1.000,1.000,1.000
"Same hour: logistic, all KPIs",1.000,1.000,1.000
"Same hour: logistic, without NEFT",1.000,1.000,1.000
Early warning: previous hour NEFT < 95%,0.000,0.000,0.000
"Early warning: logistic, lagged KPIs",0.214,0.333,0.261


**Reading:** Early warning is weak. The lagged model catches 3 of the 9 critical test hours, with 11 false alarms. The previous-hour NEFT rule catches none. Critical hours start abruptly, so the previous hour usually looks normal.

## 7. Summary

**Verified results (test period: 48 hours, 9 critical):**
- Same-hour detection: F1 = 1.000 for the threshold rule and both logistic models. This is expected because the labels are rule-based.
- Early warning: the best model has precision 0.214, recall 0.333, F1 0.261. The previous-hour rule has F1 0.000.

**Limitations:**
- One week of hourly data. No evidence about seasonality or drift.
- Labels appear to be derived from thresholds on the same KPIs.
- 26 critical hours in total, 9 in the test set. One miss changes recall by about 0.11.
- The incident file and the minute-level file disagree at every critical hour.

**Open question:** How were the `is_critical` labels created, and which file is authoritative for the critical hours?

**Future work (not in v1):** minute-level anomaly detection, log clustering on `application_logs.csv`, a RAG assistant over incident reports, a FastAPI backend, and Docker.